# Capítulo 8 — Análise e Visualização de Dados com pandas, Matplotlib e Seaborn

**Programação com Python Aplicada à Engenharia de Defesa** · IPETEC/UCP

> **Notebook de aula — Módulo IV, Sábado 4.** Ao final, o miniprojeto ganha seu **módulo de análise e visualização** — de dado bruto a gráfico de apoio à decisão.

---

### Objetivos do capítulo
Ao final, você será capaz de:
- carregar e inspecionar dados tabulares em um **DataFrame** do pandas;
- **selecionar** e **filtrar** linhas e colunas;
- resumir dados com **agregações** e `groupby`;
- trabalhar com **séries temporais** e tratar **valores ausentes**;
- produzir **gráficos** para apoio à decisão com Matplotlib e Seaborn;
- acrescentar ao **miniprojeto** um módulo de análise e visualização.

*O NumPy nos deu o cálculo sobre números. Mas os dados de defesa vêm em **tabelas**, com colunas nomeadas — sensor, tipo, velocidade, instante. O **pandas**, construído sobre o NumPy, é a ferramenta para esse tipo de dado. E, uma vez organizados, os dados precisam ser **vistos**: o **Matplotlib** e o **Seaborn** convertem números em imagens.*

> 💡 **Sobre os dados deste notebook** — Para que tudo rode de forma autônoma no Colab, a célula de **preparação** abaixo gera um pequeno conjunto de ocorrências de amostra e grava `ocorrencias.csv` e `ocorrencias.json`. Os números impressos (médias, contagens, resumos) refletem **esses dados de amostra** — portanto podem diferir dos valores específicos que aparecem no livro, que usou um conjunto maior. O que importa é que **o código de cada listagem é o mesmo do livro** e funciona de ponta a ponta.

**Preparação — cria os dados de amostra usados no capítulo.**

In [ ]:
import json
import pandas as pd

# Conjunto de amostra: 10 ocorrências ao longo de uma manhã, 3 sensores,
# incluindo um contato atípico a 120 km/h (o mesmo dos capítulos anteriores).
registros_amostra = [
    {"sensor": "Radar-A1", "tipo": "superfície", "velocidade_kmh": 22.2, "instante": "2026-05-24 08:15:00"},
    {"sensor": "Radar-A1", "tipo": "superfície", "velocidade_kmh": 44.4, "instante": "2026-05-24 08:40:00"},
    {"sensor": "Radar-B2", "tipo": "aéreo",      "velocidade_kmh": 32.4, "instante": "2026-05-24 09:05:00"},
    {"sensor": "Sonar-1",  "tipo": "submarino",  "velocidade_kmh": 18.5, "instante": "2026-05-24 09:30:00"},
    {"sensor": "Radar-B2", "tipo": "aéreo",      "velocidade_kmh": 50.0, "instante": "2026-05-24 09:55:00"},
    {"sensor": "Radar-A1", "tipo": "superfície", "velocidade_kmh": 31.3, "instante": "2026-05-24 10:20:00"},
    {"sensor": "Sonar-1",  "tipo": "submarino",  "velocidade_kmh": 35.9, "instante": "2026-05-24 10:45:00"},
    {"sensor": "Radar-B2", "tipo": "aéreo",      "velocidade_kmh": 38.3, "instante": "2026-05-24 11:10:00"},
    {"sensor": "Sonar-1",  "tipo": "submarino",  "velocidade_kmh": 27.6, "instante": "2026-05-24 11:35:00"},
    {"sensor": "Radar-A1", "tipo": "superfície", "velocidade_kmh": 120.0, "instante": "2026-05-24 12:00:00"},
]

# Grava nos dois formatos usados adiante
pd.DataFrame(registros_amostra).to_csv("ocorrencias.csv", index=False)
with open("ocorrencias.json", "w", encoding="utf-8") as f:
    json.dump(registros_amostra, f, ensure_ascii=False, indent=2)

print("Dados de amostra gravados: ocorrencias.csv e ocorrencias.json")

## 8.1 pandas: Series e DataFrame

O pandas tem duas estruturas centrais. A **Series** é uma coluna de dados com um índice; o **DataFrame** é uma tabela — um conjunto de *Series* que compartilham o mesmo índice, como uma planilha com colunas nomeadas.

Um DataFrame nasce naturalmente de uma **lista de dicionários** — exatamente a forma com que modelamos as ocorrências no Capítulo 3.

**Listagem 8.1 — Criando um DataFrame a partir de registros.**

In [ ]:
import pandas as pd

registros = [
    {"sensor": "Radar-A1", "tipo": "superfície", "velocidade_kmh": 38.3},
    {"sensor": "Radar-B2", "tipo": "aéreo",      "velocidade_kmh": 32.4},
    {"sensor": "Radar-A1", "tipo": "superfície", "velocidade_kmh": 31.3},
    {"sensor": "Sonar-1",  "tipo": "submarino",  "velocidade_kmh": 35.9},
]

df = pd.DataFrame(registros)
print(df)
print(df.shape)        # (4, 3): 4 linhas, 3 colunas

Na prática, os dados costumam vir de um arquivo. O pandas lê CSV (e muitos outros formatos) em uma linha, devolvendo um DataFrame pronto.

**Listagem 8.2 — Lendo dados de um arquivo CSV.**

In [ ]:
import pandas as pd

df = pd.read_csv("ocorrencias.csv")

print(df.head())       # as primeiras linhas
print(df.describe())   # resumo estatístico das colunas numéricas

Os métodos `head` (primeiras linhas), `shape` (dimensões) e `describe` (resumo estatístico) são os primeiros gestos de qualquer análise — a forma de "conhecer o terreno" antes de agir sobre os dados.

## 8.2 Seleção e filtragem

Acessar uma coluna é tão simples quanto indexá-la pelo nome; a filtragem usa máscaras booleanas, como no NumPy, mas agora preservando a estrutura da tabela.

**Listagem 8.3 — Selecionando colunas e filtrando linhas.**

In [ ]:
# Uma coluna (devolve uma Series)
print(df["velocidade_kmh"])

# Filtrar linhas por uma condição
alertas = df[df["velocidade_kmh"] > 40]
print(alertas)

# Combinar condições (use & para "e", | para "ou")
radar_rapido = df[(df["sensor"] == "Radar-A1") & (df["velocidade_kmh"] > 40)]
print(radar_rapido)

> ⚠️ **Armadilha comum** — Ao combinar condições no pandas, use os operadores `&` (e) e `|` (ou), **não** as palavras `and` e `or` — e envolva cada condição em **parênteses**. Escrever `df[cond1 and cond2]` provoca erro; o correto é `df[(cond1) & (cond2)]`. É uma das confusões mais comuns de quem vem do Python "puro".

## 8.3 Agregação com `groupby`

A operação mais poderosa do pandas é o `groupby`: ele **agrupa** as linhas por um critério, aplica uma agregação a cada grupo e **combina** os resultados — o padrão "dividir, aplicar, combinar". É a forma de responder a perguntas como "qual a velocidade média por sensor?".

**Listagem 8.4 — Agregações por grupo.**

In [ ]:
# Velocidade média de cada sensor
print(df.groupby("sensor")["velocidade_kmh"].mean().round(2))

# Número de ocorrências por sensor
print(df.groupby("sensor").size())

Em duas linhas, o `groupby` revela um padrão que estava oculto na tabela bruta — o tipo de descoberta que orienta uma decisão. *(Com os dados de amostra, o `Radar-A1` puxa a média para cima por conta do contato atípico a 120 km/h.)*

## 8.4 Séries temporais

Dados de monitoramento têm, quase sempre, uma dimensão temporal. O pandas trata datas e horas como um tipo de primeira classe. Convertendo uma coluna para data e usando-a como índice, abrimos acesso à **reamostragem** (*resampling*), que reagrupa os dados em intervalos de tempo.

**Listagem 8.5 — Trabalhando com uma série temporal.**

In [ ]:
# Converte a coluna de texto em datas e a torna o índice
df["instante"] = pd.to_datetime(df["instante"])
df = df.set_index("instante")

# Reamostra: velocidade média a cada 1 hora
media_horaria = df["velocidade_kmh"].resample("1h").mean()
print(media_horaria.round(1))

O `resample("1h")` agrupa as leituras por hora; poderia ser por dia (`"1D"`), por minuto (`"1min"`) ou qualquer outro intervalo. É a ferramenta para suavizar séries ruidosas e revelar tendências.

## 8.5 Valores ausentes

Dados reais são imperfeitos: um sensor falha, um campo vem em branco, uma leitura se perde. O pandas representa esses buracos com o valor especial `NaN` (*Not a Number*) e oferece ferramentas para lidar com eles — decisão que, em engenharia, é do analista.

**Listagem 8.6 — Detectando e tratando valores ausentes.**

*(Para demonstrar, criamos aqui uma cópia com uma leitura ausente de propósito.)*

In [ ]:
import numpy as np

# Cópia dos dados com um valor ausente injetado, só para o exemplo
df_falhas = df.reset_index().copy()
df_falhas.loc[2, "velocidade_kmh"] = np.nan

# Quantos valores ausentes há em cada coluna?
print(df_falhas.isna().sum())

# Opção 1: descartar as linhas com valores ausentes
df_limpo = df_falhas.dropna()

# Opção 2: preencher com um valor (p. ex., a média da coluna)
media = df_falhas["velocidade_kmh"].mean()
df_preenchido = df_falhas.fillna({"velocidade_kmh": media})

print(f"Linhas: original {len(df_falhas)}, sem NaN {len(df_limpo)}, "
      f"preenchido {len(df_preenchido)}")

> ✅ **Boa prática** — Não há resposta única para valores ausentes: descartar linhas (`dropna`) é seguro quando são poucas; preencher (`fillna`) preserva o volume, mas introduz uma estimativa. A escolha depende do problema e deve ser **consciente e documentada** — em uma análise auditável, ocultar silenciosamente a ausência de um dado é mais perigoso do que reconhecê-la.

## 8.6 Visualização com Matplotlib

Um número resume; um gráfico revela. O **Matplotlib** é a biblioteca fundamental de visualização do Python. Sua estrutura básica é simples: cria-se uma figura e um eixo (`ax`), desenha-se sobre o eixo e ajustam-se rótulos e título.

Comecemos por um **gráfico de barras** — ideal para comparar categorias, como o número de ocorrências por sensor.

**Listagem 8.7 — Um gráfico de barras com Matplotlib.**

In [ ]:
import matplotlib.pyplot as plt

contagem = df.groupby("sensor").size()

fig, ax = plt.subplots(figsize=(6, 3))
contagem.plot(kind="bar", color="#122A4A", ax=ax)
ax.set_xlabel("Sensor")
ax.set_ylabel("Número de ocorrências")
ax.set_title("Ocorrências por sensor")
plt.tight_layout()
plt.savefig("ocorrencias_por_sensor.pdf")   # salva em arquivo
plt.show()

O método `savefig` grava o gráfico em um arquivo — PDF para relatórios, PNG para a tela — e `show` o exibe. Em um notebook do Colab, o gráfico aparece logo abaixo da célula.

Para dados ao longo do tempo, o **gráfico de linha** é a escolha natural. A linha de referência no limite de alerta faz o contato atípico a 120 km/h saltar aos olhos.

**Listagem 8.8 — Um gráfico de linha com linha de referência.**

In [ ]:
import matplotlib.pyplot as plt

serie = df["velocidade_kmh"]            # indexada pelo instante

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(serie.index, serie.values, color="#B78A2A", marker="o")
ax.axhline(40, color="#A03426", linestyle="--", label="limite de alerta")
ax.set_xlabel("Instante")
ax.set_ylabel("Velocidade (km/h)")
ax.set_title("Velocidade ao longo do tempo")
ax.legend()
plt.tight_layout()
plt.show()

> 🛡️ **Contexto de defesa** — Um gráfico bem construído é um instrumento de decisão. A linha de referência no limite de alerta transforma uma série de números em uma leitura imediata: **onde** e **quando** o limite foi ultrapassado. Em um centro de operações, é essa clareza visual — e não a tabela de dados brutos — que sustenta a ação rápida. Visualizar é parte do trabalho de engenharia, não um enfeite.

## 8.7 Seaborn: estatística com elegância

O **Seaborn** é construído sobre o Matplotlib e se especializa em gráficos estatísticos, com aparência refinada e menos código. Ele entende DataFrames diretamente: basta indicar as colunas. Um **boxplot**, por exemplo, resume a distribuição de uma variável por categoria — mediana, dispersão e valores atípicos, tudo de uma vez.

**Listagem 8.9 — Um boxplot por categoria com Seaborn.**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
sns.boxplot(data=df, x="tipo", y="velocidade_kmh", ax=ax)
ax.set_xlabel("Tipo de contato")
ax.set_ylabel("Velocidade (km/h)")
ax.set_title("Distribuição de velocidade por tipo")
plt.tight_layout()
plt.show()

Cada caixa mostra onde se concentram as velocidades de um tipo de contato; o ponto isolado é o valor atípico que já conhecemos. Em uma única imagem, comparam-se as distribuições dos três tipos — algo que exigiria várias tabelas para ser percebido.

## 8.8 Miniprojeto: o módulo de análise (Módulo 4)

Damos ao miniprojeto o seu último componente antes do Projeto Final: um módulo que carrega as ocorrências persistidas, resume-as com o pandas e produz um gráfico de apoio à decisão. Reaproveitamos o **JSON** gravado pela classe do Capítulo 5 e o transformamos em um DataFrame.

**Listagem 8.10 — Miniprojeto, Módulo 4: análise e gráfico das ocorrências.**

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

def carregar_dataframe(caminho="ocorrencias.json"):
    """Lê o JSON de ocorrências e devolve um DataFrame."""
    with open(caminho, "r", encoding="utf-8") as f:
        dados = json.load(f)
    return pd.DataFrame(dados)

def relatorio(df, limite=40.0):
    """Resume as ocorrências e gera um gráfico por sensor."""
    total = len(df)
    em_alerta = (df["velocidade_kmh"] > limite).sum()
    print(f"Total de ocorrências: {total}")
    print(f"Em alerta (> {limite} km/h): {em_alerta}")
    print("Velocidade média por sensor:")
    print(df.groupby("sensor")["velocidade_kmh"].mean().round(1))

    # Gráfico: ocorrências por sensor
    contagem = df.groupby("sensor").size()
    fig, ax = plt.subplots(figsize=(6, 3))
    contagem.plot(kind="bar", color="#122A4A", ax=ax)
    ax.set_title("Ocorrências por sensor")
    ax.set_ylabel("Número de ocorrências")
    plt.tight_layout()
    plt.savefig("relatorio_ocorrencias.pdf")
    plt.show()

# ---- Uso ----
df_relatorio = carregar_dataframe()
relatorio(df_relatorio)

Com este módulo, o miniprojeto está completo em sua estrutura: **modela** os dados (Módulo 1), **persiste**-os e os organiza em objetos (Módulo 2), oferece uma **interface** e **calcula** sobre eles (Módulo 3) e, agora, **analisa** e **visualiza** (Módulo 4). De um amontoado de leituras, chegamos a um sistema que transforma dado bruto em informação para a decisão.

> 📝 **Nota** — O que falta não é mais *construir*, e sim *consolidar*. O Capítulo 9, o Projeto Final, orienta como reunir tudo em uma aplicação documentada e apresentável, e como adaptá-la ao problema da sua realidade profissional.

## 8.9 O caminho à frente

Com a análise e a visualização, encerra-se o percurso técnico do livro. O **Capítulo 9** é de natureza diferente: não introduz novas bibliotecas, mas orienta a integração de tudo no **projeto final** do curso — transformar o miniprojeto em uma aplicação própria, completa e apresentável.

## 8.10 Resumo do capítulo
- O **DataFrame** do pandas é uma tabela de colunas nomeadas; nasce de listas de dicionários ou de arquivos (`pd.read_csv`). Inspecione-o com `head`, `shape` e `describe`.
- A **filtragem** usa máscaras booleanas; combine condições com `&` e `|` entre parênteses, nunca `and`/`or`.
- O **groupby** aplica o padrão "dividir, aplicar, combinar", respondendo a perguntas por grupo.
- **Séries temporais** usam datas como índice e permitem *reamostrar* (`resample`) os dados em intervalos.
- **Valores ausentes** (`NaN`) são tratados com `dropna` ou `fillna` — uma decisão consciente e documentada.
- O **Matplotlib** produz gráficos (barras, linhas) com `savefig` para arquivo; o **Seaborn** oferece gráficos estatísticos elegantes, como o *boxplot*, a partir de DataFrames.
- No **Módulo 4**, o miniprojeto ganhou um módulo de análise: carrega o JSON, resume com o pandas e gera gráficos de apoio à decisão.

## Armadilhas comuns
- **Usar `and`/`or` ao filtrar.** No pandas, combine condições com `&` e `|`, cada uma entre parênteses.
- **Ignorar valores ausentes.** Verifique com `isna().sum()` antes de calcular; um `NaN` não tratado contamina médias e somas.
- **Esquecer de converter datas.** Texto não é data; use `pd.to_datetime` antes de operar com séries temporais.
- **Não rotular os gráficos.** Um gráfico sem título e sem rótulos de eixos é ininteligível; rotule sempre.
- **Confundir `savefig` e `show`.** Chame `savefig` **antes** de `show`; depois de exibir, a figura pode ser limpa e o arquivo sair em branco.

## Exercícios

### Essencial — fixação
**Ex. 8.1** Crie um DataFrame a partir de uma lista de pelo menos cinco dicionários de ocorrências (chaves `sensor` e `velocidade_kmh`). Imprima as primeiras linhas com `head` e o resumo com `describe`.

In [ ]:
# Ex. 8.1
import pandas as pd

registros = [
    {"sensor": "Radar-A1", "velocidade_kmh": 22.2},
    {"sensor": "Radar-B2", "velocidade_kmh": 50.0},
    {"sensor": "Sonar-1",  "velocidade_kmh": 18.5},
    {"sensor": "Radar-A1", "velocidade_kmh": 44.4},
    {"sensor": "Radar-B2", "velocidade_kmh": 38.3},
]
df_ex = pd.DataFrame(registros)
print(df_ex.head())
print(df_ex.describe())

**Ex. 8.2** A partir desse DataFrame, filtre e imprima apenas as ocorrências com velocidade acima de 40 km/h. Em seguida, imprima quantas são.

In [ ]:
# Ex. 8.2
acima = df_ex[df_ex["velocidade_kmh"] > 40]
print(acima)
print("Quantas:", len(acima))

**Ex. 8.3** Use `groupby` para calcular e imprimir a velocidade média por sensor.

In [ ]:
# Ex. 8.3
print(df_ex.groupby("sensor")["velocidade_kmh"].mean().round(2))

### Tático — aplicação
**Ex. 8.4** Carregue um CSV de ocorrências com `pd.read_csv`, verifique com `isna().sum()` se há valores ausentes na coluna de velocidade e trate-os preenchendo com a média da coluna.

In [ ]:
# Ex. 8.4
import pandas as pd

df8 = pd.read_csv("ocorrencias.csv")   # criado na célula de preparação
print("Ausentes por coluna:")
print(df8.isna().sum())

media = df8["velocidade_kmh"].mean()
df8 = df8.fillna({"velocidade_kmh": media})
print("Após preencher, ausentes na velocidade:", df8["velocidade_kmh"].isna().sum())

**Ex. 8.5** Construa um gráfico de barras, com Matplotlib, que mostre a **velocidade média** por sensor (e não a contagem). Rotule os eixos, dê título e salve em PDF.

In [ ]:
# Ex. 8.5
import pandas as pd
import matplotlib.pyplot as plt

df8 = pd.read_csv("ocorrencias.csv")
media_por_sensor = df8.groupby("sensor")["velocidade_kmh"].mean()

fig, ax = plt.subplots(figsize=(6, 3))
media_por_sensor.plot(kind="bar", color="#122A4A", ax=ax)
ax.set_xlabel("Sensor")
ax.set_ylabel("Velocidade média (km/h)")
ax.set_title("Velocidade média por sensor")
plt.tight_layout()
plt.savefig("velocidade_media_por_sensor.pdf")
plt.show()

**Ex. 8.6** Usando o Seaborn, gere um histograma (`sns.histplot`) da coluna de velocidades e descreva, em uma frase, o que a forma da distribuição revela.

In [ ]:
# Ex. 8.6
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df8 = pd.read_csv("ocorrencias.csv")

fig, ax = plt.subplots(figsize=(6, 3))
sns.histplot(data=df8, x="velocidade_kmh", bins=8, ax=ax)
ax.set_xlabel("Velocidade (km/h)")
ax.set_title("Distribuição das velocidades")
plt.tight_layout()
plt.show()

# A maioria das leituras concentra-se abaixo de 50 km/h; o contato a 120 km/h
# aparece isolado na cauda direita — visualmente, um valor atípico.

### Estratégico — extensão criativa
**Ex. 8.7** Estenda o módulo de análise (Listagem 8.10) com uma função que produza um **gráfico de linha** da velocidade média por hora (use `resample`), com uma linha de referência no limite de alerta. Reflita sobre como esse gráfico ajudaria um operador a identificar *quando* a atividade se intensifica.

In [ ]:
# Ex. 8.7
import pandas as pd
import matplotlib.pyplot as plt

def grafico_media_horaria(caminho="ocorrencias.json", limite=40.0):
    df = pd.read_json(caminho)
    df["instante"] = pd.to_datetime(df["instante"])
    serie = df.set_index("instante")["velocidade_kmh"].resample("1h").mean()

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(serie.index, serie.values, marker="o", color="#B78A2A")
    ax.axhline(limite, color="#A03426", linestyle="--", label="limite de alerta")
    ax.set_xlabel("Hora")
    ax.set_ylabel("Velocidade média (km/h)")
    ax.set_title("Velocidade média por hora")
    ax.legend()
    plt.tight_layout()
    plt.show()

grafico_media_horaria()

**Ex. 8.8** Combine tudo em um pequeno **relatório automático**: uma função que carrega as ocorrências do JSON, calcula as principais estatísticas (total, em alerta, média por sensor), gera dois gráficos (barras por sensor e série temporal) e grava um resumo em texto. Reflita sobre como esse relatório, gerado com um clique, materializa a ideia de um "sistema de apoio à decisão".

In [ ]:
# Ex. 8.8
import json
import pandas as pd
import matplotlib.pyplot as plt

def relatorio_completo(caminho="ocorrencias.json", limite=40.0):
    df = pd.read_json(caminho)

    # --- Estatísticas ---
    total = len(df)
    em_alerta = int((df["velocidade_kmh"] > limite).sum())
    media_sensor = df.groupby("sensor")["velocidade_kmh"].mean().round(1)

    linhas = [
        "RELATÓRIO DE OCORRÊNCIAS",
        f"Total: {total}",
        f"Em alerta (> {limite} km/h): {em_alerta}",
        "Velocidade média por sensor:",
        media_sensor.to_string(),
    ]
    with open("relatorio.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(linhas))

    # --- Gráfico 1: barras por sensor ---
    fig, ax = plt.subplots(figsize=(6, 3))
    df.groupby("sensor").size().plot(kind="bar", color="#122A4A", ax=ax)
    ax.set_title("Ocorrências por sensor"); ax.set_ylabel("Nº de ocorrências")
    plt.tight_layout(); plt.savefig("rel_barras.pdf"); plt.show()

    # --- Gráfico 2: série temporal ---
    df["instante"] = pd.to_datetime(df["instante"])
    serie = df.set_index("instante")["velocidade_kmh"]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(serie.index, serie.values, marker="o", color="#B78A2A")
    ax.axhline(limite, color="#A03426", linestyle="--", label="limite")
    ax.set_title("Velocidade ao longo do tempo"); ax.legend()
    plt.tight_layout(); plt.savefig("rel_serie.pdf"); plt.show()

    print("\n".join(linhas))
    print("\n(Arquivos gravados: relatorio.txt, rel_barras.pdf, rel_serie.pdf)")

relatorio_completo()

---

*Fim do Capítulo 8 e do percurso técnico. O Capítulo 9 é o **Projeto Final**: reunir tudo em uma aplicação própria, ambientada no seu problema profissional.*